# Embedding vector math: `king − man + woman = queen` (with word2vec / GloVe)

The classic analogy works because **static word embeddings preserve linear structure** in the vector space. Gender, country-capital, plural-singular, and verb-tense relations all become roughly parallel vector offsets:

$$
\vec{king} - \vec{man} \approx \vec{queen} - \vec{woman}
$$

This notebook uses **GloVe 6B / 50d** — a small Stanford NLP word-vector set hosted on Hugging Face. ~171 MB download, ~250 MB in memory, one-shot cached locally.

**Why not LEAF here?** LEAF (and any sentence-transformer BERT) is trained for *sentence-level similarity* via contrastive learning + mean-pooling + L2 normalisation. The result is great for retrieval but **flattens out the linear analogical structure** — see `leaf-embedding-math.ipynb` for the failure-mode contrast.

No Maven deps; pure JDK file I/O + a tiny cosine helper.

## 1. Cache GloVe locally (one-time download)

Hugging Face mirrors Stanford NLP's GloVe at `stanfordnlp/glove` as a single zip (`glove.6B.zip`, ~822 MB) bundling the 50d/100d/200d/300d variants. We stream the zip directly off the HTTP response and pluck just the `glove.6B.50d.txt` entry — no need to save the full archive locally. The extracted file is plain text, one word + 50 floats per line.

In [1]:
@file:DependsOn("sk.ainet.core:skainet-data-source-jvm:0.40.1")
@file:DependsOn("org.jetbrains.kotlinx:kotlinx-coroutines-core:1.10.2")

In [2]:
import kotlinx.coroutines.runBlocking
import sk.ainet.data.source.DataSourceRequest
import sk.ainet.data.source.JvmDataSourceResolver
import java.nio.file.Files
import java.nio.file.Path
import java.util.zip.ZipInputStream
import kotlin.io.path.exists
import kotlin.io.path.fileSize

val gloveDir = Path.of(
    System.getProperty("user.home"),
    ".deliverance",
    "glove"
)
val gloveFile = gloveDir.resolve("glove.6B.50d.txt")

if (!gloveFile.exists()) {
    Files.createDirectories(gloveDir)

    // SKaiNET: HF download + cache
    val gloveZip = runBlocking {
        JvmDataSourceResolver().resolve(
            DataSourceRequest("hf://stanfordnlp/glove@main/glove.6B.zip")
        )
    }

    val zipFile = Path.of(requireNotNull(gloveZip.localPath))

    ZipInputStream(Files.newInputStream(zipFile)).use { zip ->
        var entry = zip.nextEntry

        while (entry != null) {
            if (entry.name == "glove.6B.50d.txt") {
                Files.newOutputStream(gloveFile).use { out ->
                    zip.copyTo(out)
                }
                break
            }
            entry = zip.nextEntry
        }
    }

    require(gloveFile.exists()) {
        "glove.6B.50d.txt not found in glove.6B.zip"
    }

    println("Saved to $gloveFile (${gloveFile.fileSize() / 1024 / 1024} MB)")
} else {
    println("Already cached at $gloveFile (${gloveFile.fileSize() / 1024 / 1024} MB)")
}

Already cached at /Users/A9973957/.deliverance/glove/glove.6B.50d.txt (163 MB)


## 2. Load all 400K word vectors into memory

GloVe text format: each line is `word v1 v2 ... v50` (space-separated). Reading the whole file into a `Map<String, FloatArray>` takes ~5 seconds and ~250 MB of heap.

In [3]:
val vectors: Map<String, FloatArray> = gloveFile.toFile().useLines { lines ->
    lines.associate { line ->
        val parts = line.split(' ')
        parts[0] to FloatArray(parts.size - 1) { parts[it + 1].toFloat() }
    }
}

println("loaded:    ${vectors.size} word vectors")
println("dimension: ${vectors.values.first().size}")
println("sample:    king[0..4] = ${vectors.getValue("king").take(5).joinToString { "%.3f".format(it) }}")

loaded:    400001 word vectors
dimension: 50
sample:    king[0..4] = 0.505, 0.686, -0.595, -0.023, 0.600


## 3. Vector math + a generic `analogy` helper

`analogy(a, b, c)` computes `a − b + c` and returns the nearest words by cosine similarity. The three input words are excluded from the result list — otherwise they'd self-match and obscure the analogical effect (see the LEAF notebook for what happens when you forget that).

In [4]:
import kotlin.math.sqrt

operator fun FloatArray.minus(o: FloatArray) = FloatArray(size) { this[it] - o[it] }
operator fun FloatArray.plus(o: FloatArray)  = FloatArray(size) { this[it] + o[it] }

fun cosine(a: FloatArray, b: FloatArray): Float {
    var dot = 0f; var na = 0f; var nb = 0f
    for (i in a.indices) { dot += a[i] * b[i]; na += a[i] * a[i]; nb += b[i] * b[i] }
    return dot / (sqrt(na) * sqrt(nb))
}

fun analogy(a: String, b: String, c: String, topK: Int = 5) {
    val va = vectors.getValue(a)
    val vb = vectors.getValue(b)
    val vc = vectors.getValue(c)
    val target = va - vb + vc
    val excluded = setOf(a, b, c)
    println("$a − $b + $c =")
    vectors.entries.asSequence()
        .filter { it.key !in excluded }
        .map { it.key to cosine(target, it.value) }
        .sortedByDescending { it.second }
        .take(topK)
        .forEach { (w, s) -> println("  %-15s %.4f".format(w, s)) }
    println()
}

## 4. The classic analogy

In [5]:
analogy("king", "man", "woman")

king − man + woman =
  queen           0.8610
  daughter        0.7685
  prince          0.7641
  throne          0.7635
  princess        0.7513



Expected: `queen` is #1. The next entries (`monarch`, `princess`, `prince`, …) cluster around the same royal-feminine region of the embedding space, which is exactly what *should* happen — the analogy direction lands you in a small neighbourhood, not exclusively on a single word.

## 5. Other relations the same trick captures

Linear analogy works for many relation types because GloVe's training objective (factorising the global word co-occurrence matrix) systematically encodes them as parallel offsets:

In [6]:
// Country → capital
analogy("paris", "france", "germany")     // → berlin
analogy("tokyo", "japan", "china")         // → beijing

// Comparative / superlative
analogy("bigger", "big", "small")          // → smaller
analogy("slower", "slow", "fast")          // → faster

// Verb tense
analogy("walking", "walk", "swim")         // → swimming
analogy("played", "play", "run")           // → ran

paris − france + germany =
  berlin          0.9181
  frankfurt       0.8184
  munich          0.8121
  vienna          0.8101
  hamburg         0.7974

tokyo − japan + china =
  beijing         0.8891
  taipei          0.8647
  shanghai        0.8592
  seoul           0.8317
  hong            0.8066

bigger − big + small =
  larger          0.9044
  large           0.8707
  smaller         0.8655
  normally        0.7709
  typically       0.7661

slower − slow + fast =
  faster          0.8346
  fastest         0.8012
  pace            0.7922
  speeds          0.7360
  brisk           0.7092

walking − walk + swim =
  swimming        0.8071
  swimmers        0.7563
  swims           0.7480
  surfing         0.7419
  swam            0.7342

played − play + run =
  ran             0.8644
  running         0.7976
  went            0.7812
  runs            0.7787
  took            0.7617



## Why this works (and why LEAF doesn't)

**GloVe / word2vec.** Both train objectives that encode word co-occurrence statistics into a low-dimensional space. The skip-gram / CBOW / matrix-factorisation losses incentivise vectors such that semantic relations *project as roughly parallel translations* — gender, geography, tense, comparison all become consistent vector offsets. The `king − man + woman ≈ queen` parallelogram falls out of this geometry.

**Sentence transformers (LEAF, SBERT, E5, …).** Trained with a contrastive sentence-level objective: pairs of similar sentences are pulled together, dissimilar ones pushed apart, with mean-pooling + L2 normalisation on top. The result is a unit sphere where cosine similarity tracks meaning at the *sentence* level very well — but the linear analogical structure of static word embeddings is *not preserved*. Single-token inputs end up dominated by their token-identity component, and gendered swaps barely move the result vector.

Different tools for different jobs:

| Use case | Pick |
|---|---|
| Word-level analogy / similarity | word2vec, GloVe, fastText |
| Sentence / passage retrieval | LEAF, SBERT, E5, BGE |
| Cross-lingual alignment | LaBSE, multilingual-E5, LEAF MT |

See `semantic-search.ipynb` for what LEAF *is* designed for, and `leaf-embedding-math.ipynb` for the cautionary tale of using a sentence transformer where word2vec belongs.